# BÀI THỰC HÀNH 4: MẠNG NEURAL HỒI QUY

MSSV: 23520078  
Họ tên: Trần Nhật Phương Anh

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses.

Bộ dữ liệu sử dụng: [PhoMT](https://drive.google.com/drive/folders/186OAOuSEYEDVcry7WP5UBdqECXo26QAb?usp=drive_link).

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import unicodedata
from underthesea import word_tokenize as vi_tokenize
import re
import nltk
from nltk.tokenize import word_tokenize as en_tokenize
from tqdm import tqdm
from collections import Counter
from rouge import Rouge

nltk.download('punkt')

[nltk_data] Downloading package punkt to C:\Users\Phuong
[nltk_data]     Anh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
train = pd.read_json('small-PhoMT/small-train.json')
dev = pd.read_json('small-PhoMT/small-dev.json')
test = pd.read_json('small-PhoMT/small-test.json')

In [3]:
train

,english,vietnamese
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ..."
...,...,...
19995,And the man was incredibly curious .,Và người đàn ông này cực kỳ tò mò .
19996,And he wanted to understand what it was and wh...,Và ông muốn hiểu nó là gì và tại sao thế rằng ...
19997,"And one day , we were walking .","Một ngày nọ , chúng tôi đang đi bộ ."
19998,"We were in France , in Les Houches .","Chúng tôi ở Pháp , tại Les Houches ."


In [4]:
dev

,english,vietnamese
0,"﻿Hurricane Dorian , one of the most powerful s...","Vào chủ nhật ngày 1-9-2019 , cơn bão Dorian , ..."
1,Dorian is especially dangerous due to its slow...,Bão Dorian đặc biệt nguy hiểm vì nó di chuyển ...
2,"The storm passed by the Leeward Islands , Puer...","Khi đi qua quần đảo Leeward , Puerto Rico và q..."
3,The United States branch office continues to g...,Văn phòng chi nhánh Hoa Kỳ tiếp tục cập nhật t...
4,"At this time , there have been no reported inj...","Theo báo cáo đến thời điểm hiện tại , trong 46..."
...,...,...
1995,You could even imagine a version of this scena...,Một dị bản của viễn cảnh này là nơi mà mọi ngư...
1996,"Over the course of last year , open - source h...","Năm ngoái , các hacker chuyên về ổ cứng mã ngu..."
1997,"At the other end of the network , there 'd be ...","Ở đầu kia của mạng lưới , sẽ có dịch vụ giúp c..."
1998,"Now , you Web 2.0 folks in the audience know w...",Những ai ở đây thuộc thế hệ Web 2.0 sẽ hiểu tô...


In [5]:
test

,english,vietnamese
0,"Brother Albert Barnett and his wife , Sister S...","Anh Albert Barnett và chị Susan Barnett , thuộ..."
1,Severe storms ripped through parts of the sout...,"Ngày 11 và 12-1-2020 , những cơn bão lớn đã qu..."
2,"Two days of heavy rain , high winds , and nume...",Những trận mưa to và gió lớn trong suốt hai ng...
3,"Sadly , Brother Albert Barnett and his wife , ...","Đáng buồn là anh Albert Barnett 85 tuổi , và v..."
4,The United States branch also reports that at ...,Chi nhánh Hoa Kỳ cũng cho biết có ít nhất bốn ...
...,...,...
1995,Toyota applied the principles of modularity of...,Toyota áp dụng các nguyên tắc của tính đơn lẻ ...
1996,"Now fortunately , few companies succumb to cat...",Thật may là một vài công ti không chống cự ngọ...
1997,But we do read in the newspaper every day abou...,Nhưng chúng ta đọc báo chí mỗi ngày về các côn...
1998,"How is it , then , that the consumer optics gi...",Sau đó thì gã khổng lồ tiêu dùng quang học có ...


In [6]:
print("Kích thước tập train", len(train))
print("Kích thước tập dev:", len(dev))
print("Kích thước tập test:", len(test))

Kích thước tập train 20000
Kích thước tập dev: 2000
Kích thước tập test: 2000


In [7]:
print('Các cột trong tập train:', train.columns.tolist())
print('Các cột trong tập dev:', dev.columns.tolist())
print('Các cột trong tập test:', test.columns.tolist())

Các cột trong tập train: ['english', 'vietnamese']
Các cột trong tập dev: ['english', 'vietnamese']
Các cột trong tập test: ['english', 'vietnamese']


# Data Preparation

## Chuẩn hóa text

In [8]:
def preprocess_text(text, language):
    # Chuẩn hóa Unicode
    text = unicodedata.normalize('NFC', text)
    
    # Chuyển về chữ thường
    text = text.lower()
    
    # Xóa các ký tự đặc biệt không cần thiết
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s\.\,\!\?]", "", text)
    
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    if language == "en":
        tokens = en_tokenize(text)   
    elif language == "vi":
        tokens = vi_tokenize(text)
        
    return tokens

In [9]:
for dataset in [train, dev, test]:
    tqdm.pandas()
    dataset['en_tokens'] = dataset['english'].progress_apply(lambda x: preprocess_text(x, "en"))
    dataset['vi_tokens'] = dataset['vietnamese'].progress_apply(lambda x: preprocess_text(x, "vi"))

100%|██████████| 2000/2000 [00:08<00:00, 232.43it/s]


In [10]:
train

,english,vietnamese,en_tokens,vi_tokens
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .,"[it, begins, with, a, countdown, .]","[câu chuyện, bắt đầu, với, buổi, lễ, đếm, ngượ..."
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở...","[on, august, 14th, ,, 1947, ,, a, woman, in, b...","[ngày, 14, ,, tháng, 8, ,, năm, 1947, ,, gần, ..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở...","[across, india, ,, people, hold, their, breath...","[cùng, lúc, ,, trên, khắp, đất, ấn, ,, người, ..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t...","[and, at, the, stroke, of, midnight, ,, a, squ...","[khi, đồng hồ, điểm, thời khắc, nửa đêm, ,, mộ..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ...","[these, events, form, the, foundation, of, mid...","[những, sự kiện, này, là, nền móng, tạo, nên, ..."
...,...,...,...,...
19995,And the man was incredibly curious .,Và người đàn ông này cực kỳ tò mò .,"[and, the, man, was, incredibly, curious, .]","[và, người, đàn ông, này, cực kỳ, tò mò, .]"
19996,And he wanted to understand what it was and wh...,Và ông muốn hiểu nó là gì và tại sao thế rằng ...,"[and, he, wanted, to, understand, what, it, wa...","[và, ông, muốn, hiểu, nó, là, gì, và, tại sao,..."
19997,"And one day , we were walking .","Một ngày nọ , chúng tôi đang đi bộ .","[and, one, day, ,, we, were, walking, .]","[một, ngày, nọ, ,, chúng tôi, đang, đi, bộ, .]"
19998,"We were in France , in Les Houches .","Chúng tôi ở Pháp , tại Les Houches .","[we, were, in, france, ,, in, les, houches, .]","[chúng tôi, ở, pháp, ,, tại, les, houches, .]"


In [11]:
dev

,english,vietnamese,en_tokens,vi_tokens
0,"﻿Hurricane Dorian , one of the most powerful s...","Vào chủ nhật ngày 1-9-2019 , cơn bão Dorian , ...","[hurricane, dorian, ,, one, of, the, most, pow...","[vào, chủ nhật, ngày, 192019, ,, cơn, bão, dor..."
1,Dorian is especially dangerous due to its slow...,Bão Dorian đặc biệt nguy hiểm vì nó di chuyển ...,"[dorian, is, especially, dangerous, due, to, i...","[bão, dorian, đặc biệt, nguy hiểm, vì, nó, di ..."
2,"The storm passed by the Leeward Islands , Puer...","Khi đi qua quần đảo Leeward , Puerto Rico và q...","[the, storm, passed, by, the, leeward, islands...","[khi, đi, qua, quần đảo, leeward, ,, puerto ri..."
3,The United States branch office continues to g...,Văn phòng chi nhánh Hoa Kỳ tiếp tục cập nhật t...,"[the, united, states, branch, office, continue...","[văn phòng, chi nhánh, hoa kỳ, tiếp tục, cập n..."
4,"At this time , there have been no reported inj...","Theo báo cáo đến thời điểm hiện tại , trong 46...","[at, this, time, ,, there, have, been, no, rep...","[theo, báo cáo, đến, thời điểm, hiện tại, ,, t..."
...,...,...,...,...
1995,You could even imagine a version of this scena...,Một dị bản của viễn cảnh này là nơi mà mọi ngư...,"[you, could, even, imagine, a, version, of, th...","[một, dị bản, của, viễn cảnh, này, là, nơi, mà..."
1996,"Over the course of last year , open - source h...","Năm ngoái , các hacker chuyên về ổ cứng mã ngu...","[over, the, course, of, last, year, ,, open, s...","[năm ngoái, ,, các, hacker, chuyên, về, ổ, cứn..."
1997,"At the other end of the network , there 'd be ...","Ở đầu kia của mạng lưới , sẽ có dịch vụ giúp c...","[at, the, other, end, of, the, network, ,, the...","[ở, đầu, kia, của, mạng lưới, ,, sẽ, có, dịch ..."
1998,"Now , you Web 2.0 folks in the audience know w...",Những ai ở đây thuộc thế hệ Web 2.0 sẽ hiểu tô...,"[now, ,, you, web, 2.0, folks, in, the, audien...","[những ai, ở, đây, thuộc, thế hệ, web 2.0, sẽ,..."


In [12]:
test

,english,vietnamese,en_tokens,vi_tokens
0,"Brother Albert Barnett and his wife , Sister S...","Anh Albert Barnett và chị Susan Barnett , thuộ...","[brother, albert, barnett, and, his, wife, ,, ...","[anh, albert, barnett, và, chị, susan, barnett..."
1,Severe storms ripped through parts of the sout...,"Ngày 11 và 12-1-2020 , những cơn bão lớn đã qu...","[severe, storms, ripped, through, parts, of, t...","[ngày, 11, và, 1212020, ,, những, cơn, bão, lớ..."
2,"Two days of heavy rain , high winds , and nume...",Những trận mưa to và gió lớn trong suốt hai ng...,"[two, days, of, heavy, rain, ,, high, winds, ,...","[những, trận, mưa, to, và, gió, lớn, trong, su..."
3,"Sadly , Brother Albert Barnett and his wife , ...","Đáng buồn là anh Albert Barnett 85 tuổi , và v...","[sadly, ,, brother, albert, barnett, and, his,...","[đáng, buồn, là, anh, albert, barnett, 85, tuổ..."
4,The United States branch also reports that at ...,Chi nhánh Hoa Kỳ cũng cho biết có ít nhất bốn ...,"[the, united, states, branch, also, reports, t...","[chi nhánh, hoa kỳ, cũng, cho, biết, có, ít nh..."
...,...,...,...,...
1995,Toyota applied the principles of modularity of...,Toyota áp dụng các nguyên tắc của tính đơn lẻ ...,"[toyota, applied, the, principles, of, modular...","[toyota, áp dụng, các, nguyên tắc, của, tính, ..."
1996,"Now fortunately , few companies succumb to cat...",Thật may là một vài công ti không chống cự ngọ...,"[now, fortunately, ,, few, companies, succumb,...","[thật, may, là, một vài, công ti, không, chống..."
1997,But we do read in the newspaper every day abou...,Nhưng chúng ta đọc báo chí mỗi ngày về các côn...,"[but, we, do, read, in, the, newspaper, every,...","[nhưng, chúng ta, đọc, báo chí, mỗi, ngày, về,..."
1998,"How is it , then , that the consumer optics gi...",Sau đó thì gã khổng lồ tiêu dùng quang học có ...,"[how, is, it, ,, then, ,, that, the, consumer,...","[sau, đó, thì, gã, khổng lồ, tiêu dùng, quang ..."


## Chọn độ dài padding

In [13]:
# Tính độ dài câu cho tiếng Anh và tiếng Việt
train['en_len'] = train['en_tokens'].apply(len)
train['vi_len'] = train['vi_tokens'].apply(len)
dev['en_len'] = dev['en_tokens'].apply(len)
dev['vi_len'] = dev['vi_tokens'].apply(len)
test['en_len'] = test['en_tokens'].apply(len)
test['vi_len'] = test['vi_tokens'].apply(len)

In [14]:
def ratio_under_threshold(df, col_name, threshold, name="Dataset"):
    total = len(df)
    count = (df[col_name] <= threshold).sum()
    ratio = count / total * 100
    print(f"Tỉ lệ câu {col_name} <= {threshold} từ trong {name}: {ratio:.2f}%")
    
for threshold in [50, 80, 100]:
    ratio_under_threshold(train, "en_len", threshold, "train-en")
    ratio_under_threshold(train, "vi_len", threshold, "train-vi")
    ratio_under_threshold(dev, "en_len", threshold, "dev-en")
    ratio_under_threshold(dev, "vi_len", threshold, "dev-vi")
    ratio_under_threshold(test, "en_len", threshold, "test-en")
    ratio_under_threshold(test, "vi_len", threshold, "test-vi")

Tỉ lệ câu en_len <= 50 từ trong train-en: 96.92%
Tỉ lệ câu vi_len <= 50 từ trong train-vi: 97.20%
Tỉ lệ câu en_len <= 50 từ trong dev-en: 98.85%
Tỉ lệ câu vi_len <= 50 từ trong dev-vi: 99.10%
Tỉ lệ câu en_len <= 50 từ trong test-en: 97.15%
Tỉ lệ câu vi_len <= 50 từ trong test-vi: 96.75%
Tỉ lệ câu en_len <= 80 từ trong train-en: 99.68%
Tỉ lệ câu vi_len <= 80 từ trong train-vi: 99.73%
Tỉ lệ câu en_len <= 80 từ trong dev-en: 99.95%
Tỉ lệ câu vi_len <= 80 từ trong dev-vi: 99.95%
Tỉ lệ câu en_len <= 80 từ trong test-en: 99.85%
Tỉ lệ câu vi_len <= 80 từ trong test-vi: 99.75%
Tỉ lệ câu en_len <= 100 từ trong train-en: 99.91%
Tỉ lệ câu vi_len <= 100 từ trong train-vi: 99.94%
Tỉ lệ câu en_len <= 100 từ trong dev-en: 99.95%
Tỉ lệ câu vi_len <= 100 từ trong dev-vi: 99.95%
Tỉ lệ câu en_len <= 100 từ trong test-en: 100.00%
Tỉ lệ câu vi_len <= 100 từ trong test-vi: 100.00%


Chọn max_len = 50 để tối ưu thời gian huấn luyện

## Xây dựng vocab

In [15]:
def build_vocab(token_list, min_freq = 2):
    counter = Counter([tok for seq in token_list for tok in seq])
    vocab = {}
    vocab['<pad>'] = 0
    vocab['<sos>'] = 1
    vocab['<eos>'] = 2
    vocab['<unk>'] = 3
    idx = 4
    for tok, freq in counter.items():
        if freq >= min_freq:
            vocab[tok] = idx
            idx += 1
    return vocab

en_vocab = build_vocab(train['en_tokens'].tolist(), min_freq = 2)
vi_vocab = build_vocab(train['vi_tokens'].tolist(), min_freq = 2) 

In [16]:
print("Kích thước từ điển tiếng Anh:", len(en_vocab))
print("Kích thước từ điển tiếng Việt:", len(vi_vocab))

Kích thước từ điển tiếng Anh: 10054
Kích thước từ điển tiếng Việt: 7644


## Chuyển token thành ID

In [17]:
def token_to_id(tokens, vocab, max_len = 50, add_sos_eos = False):
    ids = []
    if add_sos_eos:
        ids.append(vocab.get('<sos>'))
    for tok in tokens:
        ids.append(vocab.get(tok, vocab['<unk>']))
    if add_sos_eos:
        ids.append(vocab.get('<eos>'))
        
    if len(ids) < max_len:
        ids += [vocab.get('<pad>')] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    
    return ids

train['en_ids'] = train['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
train['vi_ids'] = train['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

dev['en_ids'] = dev['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
dev['vi_ids'] = dev['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

test['en_ids'] = test['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
test['vi_ids'] = test['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

## Tạo Dataset và DataLoader

In [18]:
class PhoMTDataset(Dataset):
    def __init__(self, src_ids, tgt_ids):
        self.src = src_ids
        self.tgt = tgt_ids
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])
    
train_dataset = PhoMTDataset(train['en_ids'].tolist(), train['vi_ids'].tolist())
dev_dataset = PhoMTDataset(dev['en_ids'].tolist(), dev['vi_ids'].tolist())
test_dataset = PhoMTDataset(test['en_ids'].tolist(), test['vi_ids'].tolist())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Huấn luyện mô hình

In [19]:
def ids_to_text(seq, id2tok):
    return " ".join([id2tok[int(i)] for i in seq if int(i) in id2tok])

In [20]:
def train_model(model, train_loader, dev_loader, optimizer, loss_fn, num_epochs=20, patience=3, device="cuda"):
    id2tok = {idx: tok for tok, idx in vi_vocab.items()}
    best_rouge = 0.0
    counter = 0
    model.to(device)
    rouge = Rouge()
    
    for epoch in range(1, num_epochs+1):
        model.train()
        train_loss = 0
        for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch}"):
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            
            logits = model(src, tgt, teacher_forcing_ratio=0.5)  
            B, T, V = logits.size()
    
            loss = loss_fn(logits.reshape(B*T, V), tgt[:,1:].reshape(B*T))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        dev_loss = 0
        rouge_scores = []
        with torch.no_grad():
            for src, tgt in dev_loader:
                src, tgt = src.to(device), tgt.to(device)
                logits = model(src, tgt, teacher_forcing_ratio=0.0)  
                B, T, V = logits.size()
                loss = loss_fn(logits.reshape(B*T, V), tgt[:,1:].reshape(B*T))
                dev_loss += loss.item()
                
                pred_ids = torch.argmax(logits, dim=-1)
                pred_texts = [ids_to_text(seq, id2tok) for seq in pred_ids]
                tgt_texts = [ids_to_text(seq, id2tok) for seq in tgt]

                for p, t in zip(pred_texts, tgt_texts):
                    scores = rouge.get_scores(p, t)
                    rouge_scores.append(scores[0]["rouge-l"]["f"])

        dev_loss /= len(dev_loader)
        avg_rouge = sum(rouge_scores) / len(rouge_scores)

        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Dev Loss = {dev_loss:.4f}, Dev ROUGE-L = {avg_rouge:.4f}")

        if avg_rouge > best_rouge:
            best_rouge = avg_rouge
            counter = 0
            torch.save(model.state_dict(), "best_model2.pt")
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping ở epoch {epoch}")
                break

# Đánh giá mô hình

In [32]:
def translate_sentence(model, src, max_len=50, device="cuda"):
    model.eval()
    src = src.unsqueeze(0).to(device) 
    encoder_outputs, (hidden, cell) = model.encoder(src)

    hidden = model._transform_bidirectional_hidden(hidden)
    cell = model._transform_bidirectional_hidden(cell)
    encoder_outputs = model.encoder_proj(encoder_outputs)

    input_t = torch.tensor([[model.sos_id]], device=device)
    outputs = []

    for _ in range(max_len):
        logits, hidden, cell, attn = model.decoder(input_t, hidden, cell, encoder_outputs)
        next_tok = logits.argmax(dim=-1)
        outputs.append(next_tok.item())
        input_t = next_tok.unsqueeze(1)
        if next_tok.item() == model.eos_id:
            break

    return outputs

In [33]:
def evaluate_rouge(model, data_loader, vi_vocab, device="cuda"):
    id2word = {idx: tok for tok, idx in vi_vocab.items()}
    rouge = Rouge()
    scores = []
    for src, tgt in data_loader:
        src, tgt = src.to(device), tgt.to(device)
        for i in range(src.size(0)):
            pred_ids = translate_sentence(model, src[i], max_len=50, device=device)
            pred_tokens = [id2word.get(idx, "<unk>") for idx in pred_ids]
            ref_tokens = [id2word.get(idx, "<unk>") for idx in tgt[i].cpu().numpy() if idx not in [vi_vocab['<pad>'], vi_vocab['<sos>'], vi_vocab['<eos>']]]
            pred_sent = " ".join(pred_tokens)
            ref_sent = " ".join(ref_tokens)
            score = rouge.get_scores(pred_sent, ref_sent)[0]['rouge-l']['f']
            scores.append(score)
    avg_score = sum(scores) / len(scores)
    print(f"ROUGE-L F1: {avg_score:.4f}")

#### Bài 2: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [23]:
class BahdanauAttention(nn.Module):
    """
    Bahdanau Attention (Additive Attention)

    Công thức:
    score(s_t, h_i) = v^T * tanh(W_q * s_t + W_k * h_i)
    """
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.hidden_size = hidden_size

        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)  # Query projection
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)  # Key projection
        self.v = nn.Linear(hidden_size, 1, bias=False)  # Scoring function

    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: [batch_size, hidden_size] - s_t
            encoder_outputs: [batch_size, src_len, hidden_size] - {h_i}

        Returns:
            context: [batch_size, hidden_size]
            attention_weights: [batch_size, src_len]
        """
        # Tính scores: score = v^T * tanh(W_q * s_t + W_k * h_i)
        query = self.W_q(decoder_hidden).unsqueeze(1)  # [batch_size, 1, hidden_size]
        keys = self.W_k(encoder_outputs)  # [batch_size, src_len, hidden_size]

        # Additive attention
        energy = torch.tanh(query + keys)  # [batch_size, src_len, hidden_size]
        scores = self.v(energy).squeeze(2)  # [batch_size, src_len]

        # Softmax
        attention_weights = torch.softmax(scores, dim=1)  # [batch_size, src_len]

        # Weighted sum
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        # [batch_size, 1, src_len] x [batch_size, src_len, hidden_size] -> [batch_size, hidden_size]

        return context, attention_weights

In [24]:
class EncoderAttention(nn.Module):
    def __init__(self, vocab_size, emb_size = 256, hidden_size = 256, num_layers = 3, pad_id = 0, dropout = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=True) # Bài báo dùng BiRNN
        
    def forward(self, src):
        emb = self.embedding(src)
        outputs, hidden = self.lstm(emb)
        return outputs, hidden

In [25]:
class DecoderAttention(nn.Module):
    """
    Decoder với Bahdanau Attention cho Bài 2
    """
    def __init__(self, vocab_size, embedding_dim=256, hidden_size=256,
                 num_layers=3, dropout=0.3, pad_id = 0):
        super(DecoderAttention, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size= embedding_dim,
            hidden_size= hidden_size,
            num_layers = num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = BahdanauAttention(hidden_size)

        # Output: kết hợp LSTM output với context vector
        self.fc = nn.Linear(2 * hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs):
        """
        Args:
            input_token: [batch_size, 1]
            hidden: [num_layers, batch_size, hidden_size]
            cell: [num_layers, batch_size, hidden_size]
            encoder_outputs: [batch_size, src_len, hidden_size]

        Returns:
            prediction: [batch_size, vocab_size]
            hidden, cell: state mới
            attention_weights: [batch_size, src_len]
        """
        embedded = self.dropout(self.embedding(input_token))  # [batch_size, 1, embedding_dim]

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [batch_size, 1, hidden_size]

        # Attention
        decoder_hidden = hidden[-1]  # Lấy hidden state lớp cuối
        context, attention_weights = self.attention(decoder_hidden, encoder_outputs)

        # Kết hợp output với context
        combined = torch.cat([output.squeeze(1), context], dim=1)  # [batch_size, hidden_size*2]
        prediction = self.fc(combined)  # [batch_size, vocab_size]

        return prediction, hidden, cell, attention_weights

In [26]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, sos_id, eos_id, pad_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_id = sos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        
        self.encoder_proj = nn.Linear(encoder.lstm.hidden_size*2, encoder.lstm.hidden_size)
        
    def _transform_bidirectional_hidden(self, h):
        num_layers = h.size(0)//2
        forward = h[0:num_layers,:,:]
        backward = h[num_layers:,:,:]
        return forward + backward

        
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        encoder_outputs, (hidden, cell) = self.encoder(src)
        
        encoder_outputs = self.encoder_proj(encoder_outputs)
        hidden = self._transform_bidirectional_hidden(hidden)
        cell   = self._transform_bidirectional_hidden(cell)

        B, T = tgt.size()
        outputs = []
        input_t = tgt[:,0].unsqueeze(1) 
        for t in range(1, T):
            logits, hidden, cell, _ = self.decoder(input_t, hidden, cell, encoder_outputs)
            outputs.append(logits.unsqueeze(1))
            use_tf = torch.rand(1).item() < teacher_forcing_ratio
            next_tok = tgt[:, t] if use_tf else logits.argmax(dim=-1)
            input_t = next_tok.unsqueeze(1)
        return torch.cat(outputs, dim=1)

In [27]:
PAD_ID = 0
SOS_ID = 1
EOS_ID = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

enc = EncoderAttention(len(en_vocab), pad_id=PAD_ID)
dec = DecoderAttention(len(vi_vocab), pad_id=PAD_ID)
model = Seq2SeqAttention(enc, dec, SOS_ID, EOS_ID, PAD_ID).to(device)

optimizer = optim.Adam(model.parameters(), lr= 0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

In [28]:
train_model(model, train_loader, dev_loader, optimizer, loss_fn, num_epochs=20, patience=3, device=device)

Epoch 1: 100%|██████████| 625/625 [1:05:19<00:00,  6.27s/it]


Epoch 1: Train Loss = 5.8971, Dev Loss = 5.7676, Dev ROUGE-L = 0.2473


Epoch 2: 100%|██████████| 625/625 [1:30:16<00:00,  8.67s/it]


Epoch 2: Train Loss = 5.4185, Dev Loss = 5.6161, Dev ROUGE-L = 0.2732


Epoch 3: 100%|██████████| 625/625 [1:26:52<00:00,  8.34s/it]


...

Epoch 15: Train Loss = 2.8763, Dev Loss = 5.8473, Dev ROUGE-L = 0.3703


Epoch 16: 100%|██████████| 625/625 [56:45<00:00,  5.45s/it]


Epoch 16: Train Loss = 2.7635, Dev Loss = 5.9392, Dev ROUGE-L = 0.3693


Epoch 17: 100%|██████████| 625/625 [57:03<00:00,  5.48s/it]


Epoch 17: Train Loss = 2.6754, Dev Loss = 6.0108, Dev ROUGE-L = 0.3697
Early stopping ở epoch 17


In [29]:
model.load_state_dict(torch.load("best_model2.pt"))

<All keys matched successfully>

In [34]:
evaluate_rouge(model, test_loader, vi_vocab, device=device)

ROUGE-L F1: 0.3519
